<a href="https://colab.research.google.com/github/sarthak-geek/SIH-2026/blob/main/Data_preparation_for_training_YOLO_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
import shutil

In [2]:
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/SIH_dataset/dataset'
yolo_path = '/content/drive/MyDrive/SIH_dataset/yolo_dataset'

Mounted at /content/drive


In [3]:
for split in ['train', 'valid', 'test']:

    os.makedirs(
        os.path.join(yolo_path,split,'images'),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(yolo_path, split, 'labels'),
        exist_ok=True
    )

print("YOLO folders created")

YOLO folders created


In [4]:
for split in ['train', 'valid', 'test']:

    print(f"\nProcessing {split} dataset...")

    # --------------------------------
    # Paths for current dataset
    # --------------------------------

    split_path = os.path.join(base_path, split)

    csv_path = os.path.join(
        split_path,
        '_annotations.csv'
    )

    images_output_path = os.path.join(
        yolo_path,
        split,
        'images'
    )

    labels_output_path = os.path.join(
        yolo_path,
        split,
        'labels'
    )

    # --------------------------------
    # Read CSV
    # --------------------------------

    labels = pd.read_csv(csv_path)

    print("CSV shape:", labels.shape)

    # --------------------------------
    # Get unique image names
    # --------------------------------

    image_names = labels['filename'].unique()

    print("Unique images:", len(image_names))

    # --------------------------------
    # Process every image
    # --------------------------------

    for img_name in image_names:

        # ==============================
        # Copy image
        # ==============================

        source_image = os.path.join(
            split_path,
            img_name
        )

        destination_image = os.path.join(
            images_output_path,
            img_name
        )

        shutil.copy2(
            source_image,
            destination_image
        )

        # ==============================
        # Get ALL annotations for image
        # ==============================

        image_annotations = labels[
            labels['filename'] == img_name
        ]

        # ==============================
        # Create YOLO label filename
        # ==============================

        label_name = (
            os.path.splitext(img_name)[0]
            + '.txt'
        )

        label_path = os.path.join(
            labels_output_path,
            label_name
        )

        # ==============================
        # Write YOLO annotations
        # ==============================

        with open(label_path, 'w') as f:

            for _, row in image_annotations.iterrows():

                # Image dimensions
                img_width = row['width']
                img_height = row['height']

                # Original bounding box
                xmin = row['xmin']
                ymin = row['ymin']
                xmax = row['xmax']
                ymax = row['ymax']

                # --------------------------------
                # Convert to YOLO format
                # --------------------------------

                x_center = (
                    (xmin + xmax) / 2
                ) / img_width

                y_center = (
                    (ymin + ymax) / 2
                ) / img_height

                box_width = (
                    xmax - xmin
                ) / img_width

                box_height = (
                    ymax - ymin
                ) / img_height

                # Pothole = class 0
                class_id = 0

                # --------------------------------
                # Write one object per line
                # --------------------------------

                f.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{box_width:.6f} "
                    f"{box_height:.6f}\n"
                )

    print(f"{split} conversion completed!")


Processing train dataset...
CSV shape: (1256, 8)
Unique images: 465
train conversion completed!

Processing valid dataset...
CSV shape: (330, 8)
Unique images: 133
valid conversion completed!

Processing test dataset...
CSV shape: (154, 8)
Unique images: 67
test conversion completed!


In [5]:
yaml_content = f"""
path: {yolo_path}

train: train/images
val: valid/images
test: test/images

names:
  0: pothole
"""

yaml_path = os.path.join(
    yolo_path,
    'data.yaml'
)

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)


path: /content/drive/MyDrive/SIH_dataset/yolo_dataset

train: train/images
val: valid/images
test: test/images

names:
  0: pothole

